# Lyden M12 — context warming 1024 → 32K + CoT traces (Kaggle 2× T4 DDP)

Five warming phases resume from the M11 ckpt (`ckpt_0009600.pt` via private
dataset `chutpaglubandahure/lyden-100m-backup` v2 — ckpt_9600 + tokenizer):

| phase | window | steps | config |
|---|---|---|---|
| A | 2048 | 750 | `configs/m12-phase-a.yaml` |
| B | 4096 | 450 | `configs/m12-phase-b.yaml` |
| C | 8192 | 220 | `configs/m12-phase-c.yaml` |
| D | 16384 | 70 | `configs/m12-phase-d.yaml` |
| E | 32768 | 130 | `configs/m12-phase-e.yaml` |

~1,620 steps ≈ 0.63B tokens (~19× M11). rope_base stays 10000 — the warming
phases themselves teach the new position range. Entry ckpt 9600 → edges
10350 / 10800 / 11020 / 11090 / **11220** (edge ckpts are unconditional).

Data: one prep (`scripts/m12_prepare_data.py`) — streaming TinyStories +
specials roster [2-hop CoT trace, long fact session @8K/@16K/@32K, short
fact session] every 100 stories; distances calibrated to the tokenizer.

Success bar: story ppl ≤ 15; 1-hop recall ≥ 0.80 at T=0 on held-out probes
at 8K/16K/32K distance; long-context ppl graceful 512→32K (no collapse);
router load_top ≤ 0.5 / entropy ≥ 1.95; 2-hop CoT recall (soft, exploratory).

Run policy: **Save Version → Save & Run All (Commit)**, GPU **T4 ×2**,
Internet ON, dataset **`chutpaglubandahure/lyden-100m-backup`** attached.
New notebook — old ones stay untouched.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import sys
print(sys.version)

In [ ]:
%%bash
set -euo pipefail
if [ ! -d /kaggle/working/Lyden/.git ]; then
  git clone https://github.com/mrrchandra/Lyden.git /kaggle/working/Lyden
else
  cd /kaggle/working/Lyden && git pull --ff-only || echo '(offline keep)'
fi
ENTRY=$(find /kaggle/input -name 'ckpt_0009600.pt' 2>/dev/null | head -1 || true)
TOKJ=$(find /kaggle/input -name 'tokenizer.json' 2>/dev/null | head -1 || true)
if [ -z "$ENTRY" ] || [ -z "$TOKJ" ]; then
  echo "[FATAL] backup dataset missing: ENTRY=$ENTRY TOKJ=$TOKJ"
  echo '       attach dataset chutpaglubandahure/lyden-100m-backup (v2=ckpt_9600) and re-run.'
  exit 1
fi
mkdir -p /kaggle/working/ckpts-m11 /kaggle/working/lyden-data/tokenizer
cp "$ENTRY" /kaggle/working/ckpts-m11/ckpt_0009600.pt
cp "$TOKJ" /kaggle/working/lyden-data/tokenizer/tokenizer.json
TOKM=$(find /kaggle/input -name 'meta.json' 2>/dev/null | head -1 || true)
[ -n "$TOKM" ] && cp "$TOKM" /kaggle/working/lyden-data/tokenizer/meta.json || true
echo "[entry] restored M11 ckpt (step 9600) from:"
echo "        $ENTRY"
echo "[entry] tokenizer from $TOKJ"
ls -la /kaggle/working/ckpts-m11/ /kaggle/working/lyden-data/tokenizer/
cd /kaggle/working/Lyden
python - <<'PYEOF'
import sys
sys.path.insert(0, 'src')
from pathlib import Path
from lyden.config import load_config
if not Path('/kaggle/working/lyden-data/tokenizer/tokenizer.json').exists():
    sys.exit('[FATAL] tokenizer missing')
c = load_config('configs/m12-phase-e.yaml')
t = c.training
print('m12 phase-e ok:', t.seq_len, 'window |', t.total_steps, 'steps |', t.batch_size, 'bs x', t.grad_accum, 'accum')
print('m12 data pipeline + m12 eval scripts ready (8K/16K/32K roster)')
PYEOF
echo '[entry] M12 guard complete'


In [ ]:
# Install mamba-ssm + causal-conv1d, fastest source first:
# 1. wheels cached in this notebook's own previous output (attached as Input)
# 2. prebuilt release wheels matched to this runtime's torch/CUDA/cpXX
# 3. source build with --no-build-isolation (pip's isolated build env has no
#    torch, hence the classic crash) — built wheels land in
#    /kaggle/working/wheels so future sessions never compile again.
import glob, os, subprocess, sys, urllib.request, torch

TV = '.'.join(torch.__version__.split('+')[0].split('.')[:2])   # e.g. '2.8'
if torch.version.cuda is None:
    raise RuntimeError(
        'no GPU in this session (torch.version.cuda is None).'
        ' Attach GPU T4 x2 on the notebook and re-run (Save & Run All).')
CU = torch.version.cuda.split('.')[0]                          # '12' or '13'
CP = f'cp{sys.version_info[0]}{sys.version_info[1]}'
print('runtime:', torch.__version__, 'cuda', CU, CP)

JOBS = [
    ('causal_conv1d', 'causal-conv1d==1.7.0',
     f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/'
     f'causal_conv1d-1.7.0+cu{CU}torch{TV}cxx11abiTRUE-{CP}-{CP}-linux_x86_64.whl'),
    ('mamba_ssm', 'mamba-ssm==2.3.2.post1',
     f'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/'
     f'mamba_ssm-2.3.2.post1%2Bcu{CU}torch{TV}cxx11abiTRUE-{CP}-{CP}-linux_x86_64.whl'),
]
for name, pkg, url in JOBS:
    cached = sorted(glob.glob(f'/kaggle/input/**/{name}-*.whl', recursive=True))
    if cached:
        print(f'>> pip install (own-output cache) {cached[0]}')
        subprocess.check_call(['pip', 'install', '-q', cached[0]])
        continue
    ok = True
    try:
        urllib.request.urlopen(urllib.request.Request(url, method='HEAD'), timeout=20)
    except Exception as e:
        ok = False
        print(f'[wheel miss] {pkg}: {e}')
    if ok:
        subprocess.check_call(['pip', 'install', '-q', url])
    else:
        os.makedirs('/kaggle/working/wheels', exist_ok=True)
        print(f'>> pip wheel --no-build-isolation {pkg} (CUDA compile, up to ~45 min)')
        subprocess.check_call(['pip', 'wheel', '--no-build-isolation', '-q',
                               '-w', '/kaggle/working/wheels', pkg])
        wheel = sorted(glob.glob(f'/kaggle/working/wheels/{name}-*.whl'))[0]
        subprocess.check_call(['pip', 'install', '-q', wheel])

from mamba_ssm.modules.mamba2 import Mamba2
blk = Mamba2(d_model=512, d_state=128, d_conv=4, expand=2, layer_idx=0, use_mem_eff_path=False).cuda()
with torch.autocast('cuda', dtype=torch.float16):   # match training semantics
    y = blk(torch.randn(2, 256, 512, device='cuda'))
print('mamba-ssm OK:', y.shape, y.dtype, torch.__version__)


In [ ]:
%%bash
set -uo pipefail
cd /kaggle/working/Lyden
python scripts/m12_prepare_data.py \
  --tokenizer-dir /kaggle/working/lyden-data/tokenizer \
  --data-dir /kaggle/working/lyden-data-m12 \
  --max-train-tokens 90000000 \
  --max-val-tokens 1000000 \
  --stories-per-special 100 2>&1 | tail -25
echo '[prep] m12 mixture ready'


In [ ]:
%%bash
# M12 warming: phases A..E sequentially, T4xD DDP.
#   phase A: cross-phase entry (--resume the M11 ckpt when starting fresh);
#            if prior m12 progress exists (12h-cap continuation), resume that instead
#   later phases: chain --resume to the previous phase's edge ckpt
#   every torchrun: up to 4 rounds (retry from latest ckpt)
set -euo pipefail
cd /kaggle/working/Lyden
GPUS=$(python -c 'import torch; print(max(1, torch.cuda.device_count()))')
if [ "$GPUS" -gt 0 ]; then ACCUM_MULT=$(( 2 / GPUS )); else ACCUM_MULT=2; fi
echo "[train] gpus=$GPUS nproc-per-node=$GPUS accum-mult=$ACCUM_MULT"

ENTRY=/kaggle/working/ckpts-m11/ckpt_0009600.pt
if [ ! -f "$ENTRY" ]; then echo "[FATAL] entry ckpt missing: $ENTRY"; exit 1; fi
if [ ! -d /kaggle/working/lyden-data-m12/shards/train ]; then echo "[FATAL] m12 data missing (prep failed?)"; exit 1; fi

mkdir -p /kaggle/working/runs
own_latest_for() {
  ls -1 /kaggle/working/runs/100m-m12-$1/ckpts/ckpt_*.pt 2>/dev/null | tail -1 || true
}
A_OWN=$(own_latest_for a)

run_phase() {
  local p=$1 prev=$2
  local RESUME_ARGS=""
  if [ "$p" = "a" ]; then
    if [ -n "$A_OWN" ]; then RESUME_ARGS="--resume $A_OWN"; else RESUME_ARGS="--resume $ENTRY"; fi
  else
    local prev_ckpt; prev_ckpt=$(own_latest_for "$prev")
    if [ -n "$prev_ckpt" ]; then
      RESUME_ARGS="--resume $prev_ckpt"
      echo "[chain] phase $p resumes previous edge: $prev_ckpt"
    else
      echo "[FATAL] cannot start phase $p: previous phase $prev has no ckpt"
      return 1
    fi
  fi
  local round
  for round in 1 2 3 4; do
    local log="phase-$p-r$round.log"
    echo "[train round $round] phase $p ${RESUME_ARGS:-(auto-resume)}"
    set +e
    (torchrun --standalone --nproc-per-node="$GPUS" \
        scripts/train.py --config "configs/m12-phase-$p.yaml" \
        --accum-mult "$ACCUM_MULT" $RESUME_ARGS) 2>&1 | tee "$log" | tail -4
    set -e
    sleep 2
    if grep -q '\[done\]' "$log"; then
      echo "[train round $round] phase $p completed"
      return 0
    fi
    echo "[train round $round] phase $p failed - retrying from latest ckpt"
  done
  echo "[train] phase $p exhausted all rounds"
  return 1
}

run_phase a a || exit 1
run_phase b a || exit 1
run_phase c b || exit 1
run_phase d c || exit 1
run_phase e d || exit 1
FINAL=$(own_latest_for e)
if [ -z "$FINAL" ]; then echo "[FATAL] no m12-e ckpt at all"; exit 1; fi
echo "[m12] chain complete - final: $FINAL"


In [ ]:
%%bash
set -uo pipefail
cd /kaggle/working/Lyden
CKPT=$(ls -1 /kaggle/working/runs/100m-m12-e/ckpts/ckpt_*.pt 2>/dev/null | tail -1 || true)
if [ -z "$CKPT" ]; then echo '[eval] no m12-e ckpt yet'; exit 0; fi
echo "[eval] $CKPT"
python scripts/m12_eval.py --config configs/m12-phase-e.yaml \
  --ckpt "$CKPT" --probe-seeds 16 --cot-probe-seeds 16 \
  --ppl-batches 12 --out /kaggle/working/eval_report.json 2>&1 | tail -80
echo "=== eval report ==="
cat /kaggle/working/eval_report.json
